# Clase 196 — Feast feature store

Definir features, generar training dataset point-in-time correct, materializar al online store, servir features con baja latencia.

Requiere: `pip install feast`.

## Setup

In [ ]:
import os, shutil, tempfile
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd, numpy as np

WORK = Path(tempfile.gettempdir()) / 'feast_demo'
if WORK.exists(): shutil.rmtree(WORK)
REPO = WORK / 'feature_repo'
(REPO / 'data').mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

## 1. Generar dataset histórico (offline store: parquet)

100 drivers × 30 días × 24 horas = 72 000 filas con features.

In [ ]:
rng = np.random.default_rng(42)
now = datetime(2026, 6, 1)
rows = []
for d_id in range(1001, 1101):
    for h in range(30 * 24):
        ts = now - timedelta(hours=h)
        rows.append({
            'driver_id': d_id,
            'event_timestamp': ts,
            'conv_rate': float(rng.beta(2, 5)),
            'acc_rate': float(rng.beta(8, 2)),
            'avg_daily_trips': int(rng.poisson(12)),
            'created': ts,
        })
df = pd.DataFrame(rows)
df.to_parquet(REPO / 'data' / 'driver_stats.parquet')
print(df.shape, '→', REPO / 'data' / 'driver_stats.parquet')

## 2. Definir el feature repo

`feature_store.yaml` (config) + `definitions.py` (entities, sources, feature views).

In [ ]:
(REPO / 'feature_store.yaml').write_text('''\
project: driver_demo
registry: data/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
''')

(REPO / 'definitions.py').write_text('''\
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

driver = Entity(name="driver", join_keys=["driver_id"])

src = FileSource(
    path="data/driver_stats.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_stats_fv = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=7),
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    source=src,
)
''')

os.chdir(REPO)
import subprocess
r = subprocess.run(['feast', 'apply'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)

## 3. Training dataset point-in-time correct

Pedimos features para 3 drivers en 3 timestamps distintos. Feast devuelve el valor vigente en cada timestamp, no el más reciente.

In [ ]:
from feast import FeatureStore
store = FeatureStore(repo_path='.')

entity_df = pd.DataFrame({
    'driver_id': [1001, 1002, 1003],
    'event_timestamp': [
        datetime(2026, 5, 15, 10, 0),
        datetime(2026, 5, 20, 14, 0),
        datetime(2026, 5, 25, 8, 0),
    ],
    'label': [1, 0, 1],
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=['driver_hourly_stats:conv_rate', 'driver_hourly_stats:acc_rate', 'driver_hourly_stats:avg_daily_trips'],
).to_df()
training_df

## 4. Materializar al online store + servir features

Copiamos los valores más recientes de offline → online. Después consultamos por driver_id con latencia <2 ms.

In [ ]:
r = subprocess.run(['feast', 'materialize-incremental', '2026-06-01T23:00:00'], capture_output=True, text=True)
print(r.stdout[-400:]); print(r.stderr[-200:])

In [ ]:
online = store.get_online_features(
    features=['driver_hourly_stats:conv_rate', 'driver_hourly_stats:avg_daily_trips'],
    entity_rows=[{'driver_id': 1001}, {'driver_id': 1042}],
).to_dict()
print(online)

# Latencia
import time
t0 = time.perf_counter()
for _ in range(100):
    store.get_online_features(
        features=['driver_hourly_stats:conv_rate'],
        entity_rows=[{'driver_id': 1001}],
    ).to_dict()
print(f'latencia media: {(time.perf_counter() - t0) * 10:.2f} ms / request')

## Ejercicio guiado

1. Agregá una segunda entidad `merchant_id` y un `FeatureView` con `avg_rating`, `n_disputes_30d`.
2. Generá un training set con features de `driver` y `merchant` simultáneamente (join automático por Feast).
3. Bajá `ttl` a `timedelta(hours=1)` y verificá que `get_online_features` empieza a devolver `None` después de re-materializar con timestamp viejo.

## Conclusiones

- **Point-in-time joins** son lo que diferencia un feature store de un `LEFT JOIN`.
- El mismo código de cliente (`get_features`) sirve para training y para serving — ahí muere el training/serving skew.
- Online store local (SQLite) es para dev. En producción: Redis/DynamoDB.